# QwerySmith v3.0 — T4 Training (Colab)

Trains the 3-seed QLoRA adapters per the pinned config. **Runtime → Change runtime type → T4 GPU.**

Everything is config-driven: the repo carries pinned YAML per seed; this notebook only executes. Hardware, package versions, adapter hashes, and final loss are captured to `train_record.json` per seed (plan §8.3: state what ran on what).

| Cell | What it does |
|---|---|
| 1 | Setup: GPU check, deps, clone repo, sync env |
| 2 | Prepare: ingest → validate → retrieve → triples (CPU, ~minutes) |
| 3 | Train all 3 seeds (~40–60 min each on T4) |
| 4 | Contract sanity check on adapter seed 1 |
| 5 | Zip artifacts back to Drive for the eval step |

In [ ]:
# ===== Cell 1: setup =====
%pip install -q uv

import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory // 2**20
print(f'GPU: {name} | VRAM: {vram} MB')
assert vram >= 14000, f'need ~16GB (T4); got {vram} MB'

# repo: replace with your fork/branch
!git clone -q https://github.com/Cyrax321/QwerySmith-1.0.git /content/qwerysmith || (cd /content/qwerysmith && git pull -q)

from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/qwerysmith'
DATA = '/content/drive/MyDrive/qwerysmith_v3'   # raw CSVs live here (Kaggle download)
import sys; sys.path.insert(0, REPO)

# link raw data into the dataset dir (the one manual step)
import shutil, os
os.makedirs(f'{REPO}/datasets/olist/raw', exist_ok=True)
for f in os.listdir(f'{DATA}/raw'):
    if f.endswith('.csv'):
        shutil.copy(f'{DATA}/raw/{f}', f'{REPO}/datasets/olist/raw/{f}')
print('raw files:', sorted(os.listdir(f'{REPO}/datasets/olist/raw')))
assert len([f for f in os.listdir(f'{REPO}/datasets/olist/raw') if f.endswith('.csv')]) == 9, 'need all 9 Olist CSVs'

In [ ]:
# ===== Cell 2: prepare (CPU) — ingest, validate, retrieve, triples =====
# NOTE: questions_v1.jsonl must already be in the repo (reviewed question
# set). If you authored on Colab instead, author + review first.
%cd {REPO}
!uv run python -m qwery_smith ingest olist
!uv run python -m qwery_smith profile olist
!uv run python -m qwery_smith validate olist
!uv run python -m qwery_smith retrieve olist
!uv run python -m qwery_smith triples olist

In [ ]:
# ===== Cell 3: train all 3 seeds =====
# ML extras first (unsloth pulls its own torch pin; ~3 min)
%pip install -q unsloth trl datasets peft transformers accelerate bitsandbytes

# GPU deps are pip-installed (Colab), so run the CLI directly with system python
!python -m qwery_smith train olist --seeds 1,2,3 --execute

In [ ]:
# ===== Cell 4: contract sanity check (adapter seed 1) =====
import json, yaml
from unsloth import FastLanguageModel

run_dirs = sorted(__import__('pathlib').Path(f'{REPO}/runs/olist').glob('train_*'))
run_dir = run_dirs[-1]
cfg = yaml.safe_load(open(run_dir / 'qlora_seed1.yaml'))

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg['base_model'],
    adapter_name=str(run_dir / 'adapters' / 'adapter_seed1'),
    max_seq_length=cfg['batch']['max_len'],
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

triples_path = f'{REPO}/datasets/olist/prepared/triples_seed42.jsonl'
triple = json.loads(open(triples_path).readline())
inputs = tokenizer(triple['prompt'] + '\n\nASSISTANT:\n', return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.8, do_sample=True)
text = tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
print(text[:600])
assert 'SQL:' in text and ('ANSWER:' in text or 'REFUSAL:' in text), 'adapter violates output contract'

In [ ]:
# ===== Cell 5: zip artifacts to Drive =====
!cd {REPO} && zip -qr /content/drive/MyDrive/qwerysmith_v3/run_{run_dir.name}.zip runs/olist/{run_dir.name} datasets/olist/prepared
print('zipped:', run_dir.name)
print('next: eval_servers.ipynb — serve the matrix and run eval + report')